# Reducing Rounding Error in Latent Text Diffusion

**Pipeline:**
1. Setup & download pretrained Cosmos checkpoints + ROCStories
2. Extract latents from frozen autoencoder
3. Train TopK SAE + Dense AE baseline
4. Evaluate: Baseline vs SAE vs Dense AE (1,000 samples)
5. Scenario D: Rounding-aware diffusion fine-tuning
6. Analysis: Floor, iterative refinement, N=128, soft rounding

**Requirements:** A100 GPU, Google Drive for persistent storage.


## 0. Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/Cosmos_SAE'

import os
for d in ['checkpoints', 'cached_latents', 'sae_checkpoints', 'evaluation_results']:
    os.makedirs(f'{DRIVE_PROJECT}/{d}', exist_ok=True)
print('Drive mounted and directories ready.')


In [ ]:
REPO_URL = 'https://github.com/dbal0503/cosmos.git'
%cd /content
!rm -rf /content/cosmos
!git clone {REPO_URL} /content/cosmos
%cd /content/cosmos


In [ ]:
!pip install -q hydra-core omegaconf timm torch-ema torch-optimi rouge-score bert-score mauve-text sacrebleu datasets evaluate

import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')


## 1. Download Checkpoints & Data


In [ ]:
!pip install -q awscli
import os

AE_CKPT = f'{DRIVE_PROJECT}/checkpoints/autoencoder-num_latents=16-wikipedia-final-128/100000.pth'
DIFF_CKPT = f'{DRIVE_PROJECT}/checkpoints/diffusion-rocstories-16-d=5-final/180000.pth'

if not os.path.exists(AE_CKPT):
    os.makedirs(os.path.dirname(AE_CKPT), exist_ok=True)
    !aws s3 cp s3://cosmos-latent-diffusion/checkpoints/autoencoder-num_latents=16-wikipedia-final-128/100000.pth "{AE_CKPT}" --region eu-north-1 --no-sign-request
else:
    print(f'AE checkpoint exists')

if not os.path.exists(DIFF_CKPT):
    os.makedirs(os.path.dirname(DIFF_CKPT), exist_ok=True)
    !aws s3 cp s3://cosmos-latent-diffusion/checkpoints/diffusion-rocstories-16-d=5-final/180000.pth "{DIFF_CKPT}" --region eu-north-1 --no-sign-request
else:
    print(f'Diffusion checkpoint exists')

!ln -sfn {DRIVE_PROJECT}/checkpoints /content/cosmos/checkpoints


In [ ]:
from datasets import load_dataset
import os
DATA_DIR = '/content/cosmos/data/rocstories'
if not os.path.exists(DATA_DIR):
    ds = load_dataset('bayes-group-diffusion/rocstories')
    ds.save_to_disk(DATA_DIR)
    print(f'Saved to {DATA_DIR}')
else:
    print(f'Dataset exists at {DATA_DIR}')


## 2. Extract Latents

Run once, then restore from Drive in future sessions.


In [ ]:
# FIRST RUN ONLY
%cd /content/cosmos
!HYDRA_FULL_ERROR=1 python extract_latents.py \
    dataset=rocstories \
    encoder.latent.num_latents=16 \
    encoder.embedding.max_position_embeddings=128 \
    decoder.latent.num_latents=16 \
    decoder.embedding.max_position_embeddings=128 \
    autoencoder.model.load_checkpoint='"autoencoder-num_latents=16-wikipedia-final-128/100000.pth"'

!cp -r /content/cosmos/cached_latents {DRIVE_PROJECT}/


In [ ]:
# SESSION RESUME — restore from Drive
!cp -r {DRIVE_PROJECT}/cached_latents /content/cosmos/
!ls -lh /content/cosmos/cached_latents/rocstories/train/


## 3. Train SAE + Dense AE Baseline (~20 min each)


In [ ]:
%cd /content/cosmos
!python train_sae.py \
    --latents_dir cached_latents/rocstories \
    --sae_type topk --expansion_factor 4 --k 64 \
    --lr 3e-4 --batch_size 4096 --num_steps 50000 \
    --output_dir sae_checkpoints/topk_4x_k64


In [ ]:
!python train_sae.py \
    --latents_dir cached_latents/rocstories \
    --sae_type dense --expansion_factor 4 \
    --lr 3e-4 --batch_size 4096 --num_steps 50000 \
    --output_dir sae_checkpoints/dense_4x

!cp -r /content/cosmos/sae_checkpoints {DRIVE_PROJECT}/


In [ ]:
# SESSION RESUME — restore SAE checkpoints
!cp -r {DRIVE_PROJECT}/sae_checkpoints /content/cosmos/


## 4. Training Curves


In [ ]:
import json, matplotlib.pyplot as plt

def plot_log(path, label):
    with open(path) as f: log = json.load(f)
    steps = [e['step'] for e in log]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(label, fontsize=14)
    for ax, key, title in zip(axes.flat, ['mse','fvu','l0','dead_frac_ema'],
                               ['MSE','FVU','L0','Dead Fraction']):
        ax.plot(steps, [e[key] for e in log]); ax.set_title(title); ax.set_xlabel('Step')
    plt.tight_layout(); plt.show()

plot_log('sae_checkpoints/topk_4x_k64/training_log.json', 'TopK SAE (4x, k=64)')
plot_log('sae_checkpoints/dense_4x/training_log.json', 'Dense AE (4x, ReLU)')


## 5. SAE Hyperparameter Sweep


In [ ]:
import subprocess
configs = [
    {'expansion_factor': 2, 'k': 32, 'name': 'topk_2x_k32'},
    {'expansion_factor': 4, 'k': 64, 'name': 'topk_4x_k64'},
    {'expansion_factor': 4, 'k': 128, 'name': 'topk_4x_k128'},
    {'expansion_factor': 8, 'k': 64, 'name': 'topk_8x_k64'},
    {'expansion_factor': 8, 'k': 128, 'name': 'topk_8x_k128'},
]
for cfg in configs:
    print(f"\nTraining: {cfg['name']}")
    subprocess.run(['python', 'train_sae.py', '--latents_dir', 'cached_latents/rocstories',
        '--sae_type', 'topk', '--expansion_factor', str(cfg['expansion_factor']),
        '--k', str(cfg['k']), '--lr', '3e-4', '--batch_size', '4096',
        '--num_steps', '50000', '--output_dir', f"sae_checkpoints/{cfg['name']}"])
!cp -r /content/cosmos/sae_checkpoints {DRIVE_PROJECT}/


In [ ]:
import json, os
print(f"{'Config':<25} {'Best FVU':>10} {'Final MSE':>12} {'Final L0':>10}")
print('-' * 60)
for cfg in configs:
    p = f"sae_checkpoints/{cfg['name']}/training_log.json"
    if not os.path.exists(p): continue
    with open(p) as f: log = json.load(f)
    print(f"{cfg['name']:<25} {min(e['fvu'] for e in log):>10.4f} {log[-1]['mse']:>12.6f} {log[-1]['l0']:>10.1f}")


## 6. Evaluate: Baseline vs SAE vs Dense AE (1,000 samples)


In [ ]:
%cd /content/cosmos
!HYDRA_FULL_ERROR=1 python evaluate_sae.py \
    --dataset rocstories \
    --autoencoder_ckpt "autoencoder-num_latents=16-wikipedia-final-128/100000.pth" \
    --diffusion_ckpt "diffusion-rocstories-16-d=5-final/180000.pth" \
    --sae_ckpt sae_checkpoints/topk_4x_k64/best.pt \
    --dense_ckpt sae_checkpoints/dense_4x/best.pt \
    --sae_type topk --expansion_factor 4 --k 64 \
    --num_texts 1000 --batch_size 32 \
    --output_dir evaluation_results
!cp -r /content/cosmos/evaluation_results {DRIVE_PROJECT}/


## 7. Scenario D — Rounding-Aware Fine-Tuning (~90 min on A100)


In [ ]:
%cd /content/cosmos
!HYDRA_FULL_ERROR=1 python train_diffusion_rounding.py \
    dataset=rocstories \
    encoder.latent.num_latents=16 \
    encoder.embedding.max_position_embeddings=128 \
    decoder.latent.num_latents=16 \
    decoder.embedding.max_position_embeddings=128 \
    autoencoder.model.load_checkpoint='"autoencoder-num_latents=16-wikipedia-final-128/100000.pth"' \
    diffusion.model.load_checkpoint='"diffusion-rocstories-16-d=5-final/180000.pth"' \
    diffusion.training.batch_size=64 training=diffusion suffix=rounding-finetune \
    +finetune_lr=1e-5 +finetune_steps=20000 +rounding_lambda=0.5
!cp -r /content/cosmos/checkpoints/diffusion*rounding-finetune* {DRIVE_PROJECT}/checkpoints/ 2>/dev/null


In [ ]:
# Evaluate Scenario D
!HYDRA_FULL_ERROR=1 python evaluate_sae.py \
    --dataset rocstories \
    --autoencoder_ckpt "autoencoder-num_latents=16-wikipedia-final-128/100000.pth" \
    --diffusion_ckpt "diffusion-rocstories-16-d=5-12-layers-rounding-finetune/20000.pth" \
    --num_texts 1000 --batch_size 32 --output_dir evaluation_results_rounding


In [ ]:
# Evaluate Scenario D + SAE combined
!HYDRA_FULL_ERROR=1 python evaluate_sae.py \
    --dataset rocstories \
    --autoencoder_ckpt "autoencoder-num_latents=16-wikipedia-final-128/100000.pth" \
    --diffusion_ckpt "diffusion-rocstories-16-d=5-12-layers-rounding-finetune/20000.pth" \
    --sae_ckpt sae_checkpoints/topk_4x_k64/best.pt \
    --sae_type topk --expansion_factor 4 --k 64 \
    --num_texts 1000 --batch_size 32 --output_dir evaluation_results_rounding_sae


## 8. Analysis

Run the trainer loader cell first, then any analysis cell.


In [ ]:
# Load trainer for all analysis cells below
import torch, sys, os
sys.path.insert(0, '/content/cosmos'); os.chdir('/content/cosmos')
from utils.hydra_utils import load_config
from diffusion_trainer import DiffusionTrainer
from utils import BatchEncoding

project_root = '/content/cosmos'
cfg = load_config(project_root, os.path.join(project_root, 'conf'), overrides=['dataset=rocstories'])
cfg.ddp.enabled = False; cfg.ddp.local_rank = 0; cfg.ddp.global_rank = 0; cfg.training = ''
cfg.encoder.latent.num_latents = 16; cfg.encoder.embedding.max_position_embeddings = 128
cfg.decoder.latent.num_latents = 16; cfg.decoder.embedding.max_position_embeddings = 128
cfg.autoencoder.model.load_checkpoint = 'autoencoder-num_latents=16-wikipedia-final-128/100000.pth'
cfg.diffusion.model.load_checkpoint = 'diffusion-rocstories-16-d=5-final/180000.pth'
cfg.diffusion.training.batch_size = 32; cfg.diffusion.training.batch_size_per_gpu = 32
trainer = DiffusionTrainer(cfg); trainer.restore_checkpoint()
device = torch.device('cuda')
ae = trainer.autoencoder; ae.encoder.to(device); ae.decoder.to(device)
print('Trainer loaded.')


In [ ]:
# N=16 autoencoder rounding floor
all_l2 = []
trainer._setup_valid_data_generator()
for i, batch in enumerate(trainer.valid_loader):
    batch = batch.to(device)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
        latents, _ = ae.get_latent(batch, bert_output_masking=False)
        norm = ae.normalize_latent(latents)
        logits = ae.decoder(ae.denormalize_latent(norm))
        tokens = logits.argmax(dim=-1)
        rb = BatchEncoding({'input_ids': tokens, 'attention_mask': torch.ones_like(tokens)}).to(device)
        re, _ = ae.get_latent(rb, bert_output_masking=False)
        all_l2.append(torch.norm(norm.float() - ae.normalize_latent(re).float(), dim=-1).mean(dim=-1).cpu())
    if len(all_l2) * all_l2[-1].shape[0] >= 1000: break
print(f'N=16 autoencoder floor: {torch.cat(all_l2).mean().item():.2f}')


In [ ]:
# Iterative refinement test
trainer._setup_valid_data_generator()
with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
    _, pred_emb, _ = trainer.generate_text_batch(batch_size=32)
    z = pred_emb.to(device); z_orig = z.clone()
    print(f"{'Step':<6} {'RE vs prev':<15} {'RE vs orig':<15}")
    print('-' * 36)
    for step in range(10):
        logits = ae.decoder(ae.denormalize_latent(z))
        tokens = logits.argmax(dim=-1)
        rb = BatchEncoding({'input_ids': tokens, 'attention_mask': torch.ones_like(tokens)}).to(device)
        zn, _ = ae.get_latent(rb, bert_output_masking=False)
        zn = ae.normalize_latent(zn)
        print(f'{step:<6} {torch.norm(z.float()-zn.float(),dim=-1).mean().item():<15.2f} {torch.norm(z_orig.float()-zn.float(),dim=-1).mean().item():<15.2f}')
        z = zn


In [ ]:
# N=128 autoencoder floor
import shutil
!aws s3 cp s3://cosmos-latent-diffusion/checkpoints/autoencoder-num_latents=128-wikipedia-final-128/100000.pth /tmp/ae128.pth --region eu-north-1 --no-sign-request
os.makedirs('/content/cosmos/checkpoints/autoencoder-num_latents=128-wikipedia-final-128/', exist_ok=True)
shutil.copy('/tmp/ae128.pth', '/content/cosmos/checkpoints/autoencoder-num_latents=128-wikipedia-final-128/100000.pth')

from encoder_trainer import EncoderTrainer
c128 = load_config(project_root, os.path.join(project_root, 'conf'), overrides=['dataset=rocstories'])
c128.ddp.enabled = False; c128.ddp.local_rank = 0; c128.ddp.global_rank = 0; c128.training = ''
c128.encoder.latent.num_latents = 128; c128.encoder.embedding.max_position_embeddings = 128
c128.decoder.latent.num_latents = 128; c128.decoder.embedding.max_position_embeddings = 128
c128.autoencoder.model.load_checkpoint = 'autoencoder-num_latents=128-wikipedia-final-128/100000.pth'
ae128 = EncoderTrainer(c128); ae128.encoder.to(device); ae128.decoder.to(device)

all_l2 = []
ae128._setup_valid_data_generator()
for i, batch in enumerate(ae128.valid_loader):
    batch = batch.to(device)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
        lat, _ = ae128.get_latent(batch, bert_output_masking=False)
        norm = ae128.normalize_latent(lat)
        logits = ae128.decoder(ae128.denormalize_latent(norm))
        tokens = logits.argmax(dim=-1)
        rb = BatchEncoding({'input_ids': tokens, 'attention_mask': torch.ones_like(tokens)}).to(device)
        re, _ = ae128.get_latent(rb, bert_output_masking=False)
        all_l2.append(torch.norm(norm.float()-ae128.normalize_latent(re).float(), dim=-1).mean(dim=-1).cpu())
    if len(all_l2) * all_l2[-1].shape[0] >= 1000: break
print(f'N=128 floor: {torch.cat(all_l2).mean().item():.2f} | N=16 floor: 33.79 | Rounding FT: 27.36')


In [ ]:
# Soft vs hard rounding
import torch.nn.functional as F
embed_matrix = ae.encoder.text_encoder.embeddings.word_embeddings.weight
all_hard, all_soft = [], []
trainer._setup_valid_data_generator()
for i, batch in enumerate(trainer.valid_loader):
    batch = batch.to(device)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
        lat, _ = ae.get_latent(batch, bert_output_masking=False)
        norm = ae.normalize_latent(lat)
        logits = ae.decoder(ae.denormalize_latent(norm))
        # Hard
        tokens = logits.argmax(dim=-1)
        rb = BatchEncoding({'input_ids': tokens, 'attention_mask': torch.ones_like(tokens)}).to(device)
        rh, _ = ae.get_latent(rb, bert_output_masking=False)
        all_hard.append(torch.norm(norm.float()-ae.normalize_latent(rh).float(), dim=-1).mean(dim=-1).cpu())
        # Soft
        sp = F.softmax(logits.float(), dim=-1)
        se = sp @ embed_matrix.float()
        bn = ae.encoder.text_encoder.embeddings.LayerNorm
        pos = ae.encoder.text_encoder.embeddings.position_embeddings(torch.arange(se.shape[1], device=device).unsqueeze(0))
        se = bn(se + pos.float())
        eo = ae.normalize_encodings(se.to(torch.bfloat16))
        rs = ae.encoder(token_ids=tokens, mask_tokens=torch.ones(se.shape[:2], device=device), token_embeddings=eo)
        all_soft.append(torch.norm(norm.float()-ae.normalize_latent(rs).float(), dim=-1).mean(dim=-1).cpu())
    if len(all_hard) * all_hard[-1].shape[0] >= 1000: break
print(f'Hard (argmax):  {torch.cat(all_hard).mean().item():.2f}')
print(f'Soft (softmax): {torch.cat(all_soft).mean().item():.2f}')
print('Conclusion: argmax is NOT the bottleneck — encoder-decoder round-trip is')


## 9. SAE Feature Analysis


In [ ]:
# Feature interpretability
from architecture.sparse_autoencoder import TopKSparseAutoencoder
sae = TopKSparseAutoencoder(d_input=768, expansion_factor=4, k=64)
state = torch.load('sae_checkpoints/topk_4x_k64/best.pt', map_location='cpu')
sae.load_state_dict(state['sae_state_dict']); sae.to(device).eval()

feature_texts = {i: [] for i in range(sae.d_hidden)}
trainer._setup_valid_data_generator()
for bi, batch in enumerate(trainer.valid_loader):
    batch = batch.to(device)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
        lat, _ = ae.get_latent(batch, bert_output_masking=False)
        _, sc, _ = sae(ae.normalize_latent(lat))
        ma = sc.max(dim=1).values
        for i in range(len(batch['text_trg'])):
            for fi in ma[i].topk(10).indices.tolist():
                v = ma[i, fi].item()
                if v > 0: feature_texts[fi].append((v, batch['text_trg'][i]))
    if bi >= 50: break

active = sorted([(len(v), k) for k, v in feature_texts.items() if len(v) > 5], reverse=True)
for count, fi in active[:15]:
    texts = sorted(feature_texts[fi], reverse=True)[:3]
    print(f'\n--- Feature {fi} ({count} activations) ---')
    for v, t in texts: print(f'  [{v:.2f}] {t[:120]}')


In [ ]:
# Features suppressed/amplified by rounding FT
import numpy as np

trainer._setup_valid_data_generator()
baseline_acts = []
with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
    for i in range(5):
        _, pe, _ = trainer.generate_text_batch(batch_size=32)
        _, sc, _ = sae(pe.to(device))
        baseline_acts.append(sc.max(dim=1).values.cpu())
baseline_acts = torch.cat(baseline_acts)

c2 = load_config(project_root, os.path.join(project_root, 'conf'), overrides=['dataset=rocstories'])
c2.ddp.enabled=False; c2.ddp.local_rank=0; c2.ddp.global_rank=0; c2.training=''
c2.encoder.latent.num_latents=16; c2.encoder.embedding.max_position_embeddings=128
c2.decoder.latent.num_latents=16; c2.decoder.embedding.max_position_embeddings=128
c2.autoencoder.model.load_checkpoint='autoencoder-num_latents=16-wikipedia-final-128/100000.pth'
c2.diffusion.model.load_checkpoint='diffusion-rocstories-16-d=5-12-layers-rounding-finetune/20000.pth'
c2.diffusion.training.batch_size=32; c2.diffusion.training.batch_size_per_gpu=32
t2 = DiffusionTrainer(c2); t2.restore_checkpoint()
t2._setup_valid_data_generator(); t2.autoencoder.decoder.to(device)

rounding_acts = []
with torch.autocast(device_type='cuda', dtype=torch.bfloat16), torch.no_grad():
    for i in range(5):
        _, pe, _ = t2.generate_text_batch(batch_size=32)
        _, sc, _ = sae(pe.to(device))
        rounding_acts.append(sc.max(dim=1).values.cpu())
rounding_acts = torch.cat(rounding_acts)

bm = baseline_acts.float().mean(dim=0).numpy()
rm = rounding_acts.float().mean(dim=0).numpy()
d = rm - bm
print('MOST SUPPRESSED:')
for i in np.argsort(d)[:10]: print(f'  Feature {i}: {bm[i]:.3f} -> {rm[i]:.3f} (d={d[i]:.3f})')
print('\nMOST AMPLIFIED:')
for i in np.argsort(d)[-10:][::-1]: print(f'  Feature {i}: {bm[i]:.3f} -> {rm[i]:.3f} (d={d[i]:.3f})')
print(f'\n>50%% suppressed: {(d < -bm*0.5).sum()} | >50%% amplified: {(d > bm*0.5).sum()}')
